In [4]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["PATH"] = "/mnt/lustre-grete/usr/u12045/projects/LLAVA-Med/envs/lerobot/bin:" + os.environ.get("PATH", "")
os.environ["HF_HOME"] = "/mnt/lustre-grete/usr/u12045/vla/hf_cache"
os.environ["TMPDIR"] = "/mnt/lustre-grete/usr/u12045/vla/cache"
os.environ["PYTHONPATH"] = "/mnt/lustre-grete/usr/u12045/vla/duci/VLA-Humanoid:" + os.environ.get("PYTHONPATH", "")

In [ ]:
import logging
import time
from contextlib import nullcontext
from pprint import pformat
from typing import Any

import torch
from termcolor import colored
from torch.amp import GradScaler
from torch.optim import Optimizer

from lerobot.common.datasets.factory import make_dataset
from lerobot.common.datasets.sampler import EpisodeAwareSampler
from lerobot.common.datasets.utils import cycle
from lerobot.common.envs.factory import make_env
from lerobot.common.optim.factory import make_optimizer_and_scheduler
from lerobot.common.policies.factory import make_policy
from lerobot.common.policies.pretrained import PreTrainedPolicy
from lerobot.common.policies.utils import get_device_from_parameters
from lerobot.common.utils.logging_utils import AverageMeter, MetricsTracker
from lerobot.common.utils.random_utils import set_seed
from lerobot.common.utils.train_utils import (
    get_step_checkpoint_dir,
    get_step_identifier,
    load_training_state,
    save_checkpoint,
    update_last_checkpoint,
)
from lerobot.common.utils.utils import (
    format_big_number,
    get_safe_torch_device,
    has_method,
    init_logging,
)
from lerobot.common.utils.wandb_utils import WandBLogger
from lerobot.configs import parser
from lerobot.configs.train import TrainPipelineConfig
from lerobot.scripts.eval import eval_policy

from dotenv import load_dotenv
load_dotenv()

%load_ext autoreload
%autoreload 2

ModuleNotFoundError: No module named 'lerobot'

In [15]:
init_logging()

In [16]:
from lerobot.configs.policies import PreTrainedConfig
from lerobot.configs.default import DatasetConfig, EvalConfig, WandBConfig

# output_dir="outputs/train/2025-06-14/05-13-17_calvin_finetune_100_vision_chunk4_aug_lr_action01_state01_imageMT_5accu"
cfg = TrainPipelineConfig(
    policy=PreTrainedConfig.from_pretrained("lerobot/pi0"),
    dataset=DatasetConfig(repo_id="IPEC-COMMUNITY/libero_spatial_no_noops_image_lerobot"),
    wandb=WandBConfig(
        project="lerobot",
        entity="lerobot",
    ),
)
cfg.validate()
logging.info(pformat(cfg.to_dict()))


INFO 2025-06-25 17:56:06 192413777.py:14 {'batch_size': 32,
 'dataset': {'episodes': None,
             'image_transforms': {'enable': False,
                                  'image_tfs': {'brightness': {'kwargs': {'brightness': [0.8,
                                                                                         1.2]},
                                                               'type': 'ColorJitter',
                                                               'weight': 1.0},
                                                'contrast': {'kwargs': {'contrast': [0.8,
                                                                                     1.2]},
                                                             'type': 'ColorJitter',
                                                             'weight': 1.0},
                                                'crop_resize': {'kwargs': {'ratio': [1,
                                                                        

In [17]:
cfg.policy.chunk_size = 4
cfg.policy.n_action_steps = 4
cfg.batch_size = 1
cfg.wandb.enable = False
cfg.policy.pretrained_path = "outputs/train/2025-06-25/10-34-13_libero_spatial_finetune_sameconfig/checkpoints/020000/pretrained_model"

In [18]:
# if cfg.wandb.enable and cfg.wandb.project:
#     wandb_logger = WandBLogger(cfg)
# else:
#     wandb_logger = None
#     logging.info(colored("Logs will be saved locally.", "yellow", attrs=["bold"]))

if cfg.seed is not None:
    set_seed(cfg.seed)

device = get_safe_torch_device(cfg.policy.device, log=True)
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

logging.info("Creating dataset")
dataset = make_dataset(cfg)

INFO 2025-06-25 17:56:07 200059321.py:14 Creating dataset
WARNING 2025-06-25 17:56:07 ts/utils.py:303 
The dataset you requested (IPEC-COMMUNITY/libero_spatial_no_noops_image_lerobot) is in 2.0 format.
While current version of LeRobot is backward-compatible with it, the version of your dataset still uses global
stats instead of per-episode stats. Update your dataset stats to the new format using this command:
```
python lerobot/common/datasets/v21/convert_dataset_v20_to_v21.py --repo-id=IPEC-COMMUNITY/libero_spatial_no_noops_image_lerobot
```

If you encounter a problem, contact LeRobot maintainers on [Discord](https://discord.com/invite/s3KuuzsPFb)
or open an [issue on GitHub](https://github.com/huggingface/lerobot/issues/new/choose).

WARNING 2025-06-25 17:56:07 ts/utils.py:303 
The dataset you requested (IPEC-COMMUNITY/libero_spatial_no_noops_image_lerobot) is in 2.0 format.
While current version of LeRobot is backward-compatible with it, the version of your dataset still uses globa

Resolving data files:   0%|          | 0/432 [00:00<?, ?it/s]

In [19]:
# create dataloader for offline training
if hasattr(cfg.policy, "drop_n_last_frames"):
    shuffle = False
    sampler = EpisodeAwareSampler(
        dataset.episode_data_index,
        drop_n_last_frames=cfg.policy.drop_n_last_frames,
        shuffle=True,
    )
else:
    shuffle = True
    sampler = None

dataloader = torch.utils.data.DataLoader(
    dataset,
    num_workers=cfg.num_workers,
    batch_size=cfg.batch_size,
    shuffle=shuffle,
    sampler=sampler,
    pin_memory=device.type != "cpu",
    drop_last=False,
)
dl_iter = cycle(dataloader)


## try to load exact the same as when training

In [20]:
# logging.info("Creating policy")
# policy = make_policy(
#     cfg=cfg.policy,
#     ds_meta=dataset.meta,
# )

In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["PATH"] = "/mnt/lustre-grete/usr/u12045/projects/LLAVA-Med/envs/lerobot/bin:" + os.environ.get("PATH", "")
os.environ["HF_HOME"] = "/mnt/lustre-grete/usr/u12045/vla/hf_cache"
os.environ["TMPDIR"] = "/mnt/lustre-grete/usr/u12045/vla/cache"
os.environ["PYTHONPATH"] = "/mnt/lustre-grete/usr/u12045/vla/duci/VLA-Humanoid:" + os.environ.get("PYTHONPATH", "")
import sys
sys.path.insert(0, "/mnt/lustre-grete/usr/u12045/vla/duci/VLA-Humanoid")


%load_ext autoreload
%autoreload 2

In [ ]:
from lerobot.common.policies.pi0.modeling_pi0 import PI0Policy
model_dir = "../outputs/train/2025-07-14/19-24-19_libero_merged_100%_defaultconfig_newcp/checkpoints/060000/pretrained_model"
policy = PI0Policy.from_pretrained(model_dir)

torch.Size([7])
Loading weights from local directory


: 

In [22]:
# policy = make_policy(
#     cfg=cfg.policy,
#     ds_meta=dataset.meta,
# )

In [23]:
# total_loss = []
# for batch in dl_iter:
#     for key in batch:
#         if isinstance(batch[key], torch.Tensor):
#             batch[key] = batch[key].to(device, non_blocking=True)
#     with torch.no_grad():
#         loss, output_dict = policy.forward(batch)
#     total_loss.append(loss.item())
#     if len(total_loss) % 10 == 0:
#         print('average loss', sum(total_loss) / len(total_loss))

In [24]:
from collections import deque
total_loss = []
for batch in dl_iter:
    print(batch.keys())
    for key in batch:
        if isinstance(batch[key], torch.Tensor):
            batch[key] = batch[key].to(device, non_blocking=True)
    with torch.no_grad():
        policy._action_queue = deque()
        pred = policy.select_action_chunk(batch)[:,0,:].unsqueeze(1)
    pred[:,:,-1] = torch.where(
            pred[:,:,-1] > 0, torch.tensor(1.0, device=device), torch.tensor(-1.0, device=device)
        )
    print(pred.shape)
    gt = batch['joint_actions'][:,0,:].unsqueeze(1)

    assert pred.shape == gt.shape, f"Pred shape {pred.shape} does not match GT shape {gt.shape}"
    loss = torch.nn.functional.mse_loss(pred, gt)
    total_loss.append(loss.item())
    if len(total_loss) % 10 == 0:
        print('pred')
        print(pred)
        print('gt')
        print(gt)
        print('average l2 loss', sum(total_loss) / len(total_loss))
        break


dict_keys(['observation.images.image', 'observation.images.wrist_image', 'observation.state', 'action', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index', 'action_is_pad', 'task'])
ductop1
tensor(-0.1127, device='cuda:0') tensor(3.1363, device='cuda:0')
tensor(-0.8230, device='cuda:0') tensor(1.3858, device='cuda:0')
torch.Size([1, 1, 7])


KeyError: 'joint_actions'

In [10]:
len(total_loss), sum(total_loss) / len(total_loss), min(total_loss), max(total_loss)

(10, 2.07571542263031, 1.157210350036621, 3.7296085357666016)